In [72]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [74]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [76]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./market-analysis-project-91130-9f9a036682b6.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [78]:

sql = f"""
select * from wook.amz_pdp_price_sales_asin_daily
"""

df = bqclient.query(sql).to_dataframe()

In [79]:
df

,crawl_date,asin,bw_price,ordered_revenue,ordered_units
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0
...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0


In [82]:
df.to_csv("amz_pdp_price_sales_daily.csv", index=False)

In [83]:
df1 = df[(df['ordered_units'] > 0) & (df['bw_price'] > 0)].copy()

In [84]:
df1

,crawl_date,asin,bw_price,ordered_revenue,ordered_units
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0
...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0


In [50]:
df1['log_units'] = np.log(df1['ordered_units'])
df1['log_price'] = np.log(df1['bw_price'])

In [52]:
# 2) inf, -inf → NaN 으로 변환
df1.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3) 로그 변수에 NaN 혹은 결측치가 있는 행 제거
df1.dropna(subset=['log_units', 'log_price'], inplace=True)

In [60]:
df1

,crawl_date,asin,bw_price,ordered_revenue,ordered_units,log_units,log_price
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0,4.356709,4.777273
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0,5.111988,5.198884
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0,5.730100,4.615022
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0,3.610918,4.950956
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0,4.672829,5.539851
...,...,...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0,2.772589,5.293305
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0,2.772589,5.863603
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0,2.772589,4.577799
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0,2.772589,4.488636


In [64]:
# 4) ASIN별 회귀 수행 및 탄력도 수집

# (2) ASIN별 회귀 및 탄력도 수집
results = []
for asin, grp in df1.groupby('asin'):
    # 관측치가 2개 미만이면 스킵
    if len(grp) < 2:
        continue
   
    if grp['log_units'].nunique() < 2:
        continue
        
    # 독립변수·종속변수 준비
    X = sm.add_constant(grp['log_price'])   # 상수항 추가
    y = grp['log_units']

    # OLS 회귀
    model = sm.OLS(y, X).fit()
    
    # 결과 추출
    results.append({
        'asin': asin,
        'elasticity': model.params['log_price'],
        'r_squared': model.rsquared,
        'n_obs': int(model.nobs)
    })


elasticity_df = pd.DataFrame(results)


In [66]:
elasticity_df

,asin,elasticity,r_squared,n_obs
0,B005G02ESA,-5.361808,0.459479,266
1,B005G5RZIY,-7.986310,0.517420,253
2,B005G5SIRG,-6.149521,0.292622,322
3,B006L9QN4G,-4.222518,0.390358,307
4,B006L9SXVW,-0.009153,0.000001,251
...,...,...,...,...
1473,B0D2VP2JDK,8.954582,0.136580,141
1474,B0DFPBM9RJ,-1.331158,0.010140,174
1475,B0DFPG616K,-1.262196,0.023505,87
1476,B0DFPGX9RJ,-4.010237,0.153522,123


In [68]:
elasticity_df = elasticity_df.sort_values('elasticity')
print(elasticity_df.head(10))     # 가장 낮은(절대값 큰 수요 민감) 10개
print(elasticity_df.tail(10))     # 가장 높은(절대값 작은 수요 민감) 10개

            asin  elasticity  r_squared  n_obs
1359  B0CSJMCXKB  -35.246640   0.026617    332
439   B074Q439NV  -30.104023   0.002589    475
1249  B0CKYRS739  -18.399467   0.058931    437
728   B088RBMBHH  -17.755451   0.110664    111
1385  B0CSK1787N  -13.656752   0.336667    265
1391  B0CSK3WS69  -12.036283   0.226314    266
1324  B0CKZ12KX4  -11.041125   0.186185    468
1313  B0CKYZHV5D  -10.953028   0.172418    466
1237  B0CJ4NMK8N  -10.410703   0.749316    169
1381  B0CSJXJZVN  -10.157179   0.327867    282
            asin  elasticity  r_squared  n_obs
1404  B0CSYBPMW4   10.657303   0.061657    246
784   B08D3C6MLR   10.888491   0.250000      3
837   B08G8MZF3Z   11.519411   0.085869      9
1470  B0D2VN15T1   11.527300   0.075367    176
1333  B0CL4FP7PP   11.675008   0.060938    301
1392  B0CSK5C8MJ   12.423801   0.035902    324
1374  B0CSJTKX7K   14.014219   0.011883    320
1342  B0CLCGMFBR   17.067017   0.146977     26
20    B006MIVR06   49.056983   0.011241     60
668   B07T9K9